# Whisper LoRA training on Kaggle

This notebook runs the engineering pilot for the Tunisian speech adapter. It freezes the baseline model and trains only q_proj/v_proj LoRA adapters. The pilot is not a scientific result.

The independent 96-utterance final benchmark is never loaded here.

In [ ]:
# Locate the repository before installing project-only dependencies.
from pathlib import Path
import os
import subprocess
import sys

candidates = [
    Path('/kaggle/working/tunisian-speech-ai'),
    Path('/kaggle/input/tunisian-speech-ai'),
]
REPO_URL = os.environ.get('REPO_URL', '')
REPO_DIR = next((path for path in candidates if (path / 'requirements.txt').exists()), None)

if REPO_DIR is None and REPO_URL:
    REPO_DIR = Path('/kaggle/working/tunisian-speech-ai')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

if REPO_DIR is None:
    raise FileNotFoundError(
        'Repository not found. Expected /kaggle/working/tunisian-speech-ai '
        'or /kaggle/input/tunisian-speech-ai, or set REPO_URL.'
    )

# Do not install requirements.txt wholesale on Kaggle: its pinned torch/numpy/
# pandas versions can replace Kaggle's CUDA stack and create unrelated conflicts.
%pip install -q --no-deps transformers==4.48.3 accelerate==1.2.1 peft==0.14.0 datasets==3.2.0 evaluate==0.4.3 jiwer==3.0.5 soundfile==0.13.0 PyYAML==6.0.2
print('Repository:', REPO_DIR)
print('Kaggle CUDA packages were preserved; only project libraries were installed.')
print('Restart the kernel once if Kaggle reports that updated packages need a restart.')

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert shutil.which('nvidia-smi'), 'nvidia-smi is unavailable on this Kaggle runtime.'
import torch
assert torch.cuda.is_available(), 'CUDA is unavailable. Enable a GPU accelerator in Kaggle.'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

## Locate the code and private dataset

Attach the project as a Kaggle Dataset named `tunisian-speech-ai`, or set `REPO_URL` to a repository URL. Attach the private dataset separately and set `DATASET_DIR` to its mounted path if needed.

In [ ]:
# Reuse the repository location discovered in the installation cell.
if 'REPO_DIR' not in globals() or not (REPO_DIR / 'requirements.txt').exists():
    raise FileNotFoundError('Run the dependency installation cell first.')

sys.path.insert(0, str(REPO_DIR))
print('Repository:', REPO_DIR)

In [ ]:
# Point this at the private Kaggle dataset containing train.jsonl and validation.jsonl.
DATASET_DIR = Path(os.environ.get('DATASET_DIR', str(REPO_DIR / 'data')))
TRAIN_MANIFEST = Path(os.environ.get('TRAIN_MANIFEST', str(DATASET_DIR / 'splits' / 'train.jsonl')))
VALIDATION_MANIFEST = Path(os.environ.get('VALIDATION_MANIFEST', str(DATASET_DIR / 'splits' / 'validation.jsonl')))

# A single train manifest is also supported; the script creates a speaker-agnostic
# pilot split only when no validation manifest is supplied.
if not TRAIN_MANIFEST.exists():
    raise FileNotFoundError(f'Train manifest not found: {TRAIN_MANIFEST}')
if not VALIDATION_MANIFEST.exists():
    VALIDATION_MANIFEST = None
    print('No validation manifest found; a temporary random pilot split will be used.')
else:
    print('Validation manifest:', VALIDATION_MANIFEST)
print('Train manifest:', TRAIN_MANIFEST)

In [ ]:
# Validate the manifest before training. This rejects missing audio, empty text,
# protected final-test markers, and train/validation speaker overlap.
validation_command = [sys.executable, str(REPO_DIR / 'src' / 'data' / 'validate_dataset.py'), str(TRAIN_MANIFEST)]
subprocess.run(validation_command, check=True)
if VALIDATION_MANIFEST is not None:
    subprocess.run([sys.executable, str(REPO_DIR / 'src' / 'data' / 'validate_dataset.py'), str(VALIDATION_MANIFEST)], check=True)

## Pilot: 100-500 examples and at most 200 steps

The initial configuration is deliberately conservative: batch size 1, gradient accumulation 8, fp16, gradient checkpointing, learning rate 1e-4, five configured epochs, and LoRA rank 8. The pilot overrides the amount of data and steps; it does not tune hyperparameters or use the final benchmark.

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/lora_pilot')
command = [
    sys.executable, str(REPO_DIR / 'scripts' / 'train_lora.py'),
    '--train-manifest', str(TRAIN_MANIFEST),
    '--output-dir', str(OUTPUT_DIR),
    '--device', 'cuda',
    '--max-train-samples', '500',
    '--max-validation-samples', '100',
    '--max-steps', '200',
    '--logging-steps', '10',
]
if VALIDATION_MANIFEST is not None:
    command.extend(['--validation-manifest', str(VALIDATION_MANIFEST)])
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
# Inspect pilot evidence. Metrics are engineering checks, not final claims.
import json
metrics_path = OUTPUT_DIR / 'metrics.json'
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2, ensure_ascii=False))
print('Saved files:')
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR))

In [ ]:
# Resume example: set this to a checkpoint directory and rerun the command.
# checkpoint = OUTPUT_DIR / 'checkpoints' / 'checkpoint-200'
# command.extend(['--resume-from-checkpoint', str(checkpoint)])

# Download the adapter and logs from the Kaggle right-side Output panel, or zip them:
archive = shutil.make_archive('/kaggle/working/lora_pilot_outputs', 'zip', OUTPUT_DIR)
print('Download archive:', archive)